# 🎯 99.9% ATTEMPT — 5-Fold Ensemble
Train 5 ResNet50 models on different data splits, ensemble all at inference.


In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms, models
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
from sklearn.model_selection import StratifiedKFold
from PIL import Image
import pandas as pd
import numpy as np
import os
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

Device: cuda
GPU: Tesla P100-PCIE-16GB


In [2]:
# ── CONFIG ─────────────────────────────────────────────────
CSV_PATH   = '/kaggle/input/image-classification-challenge-haradi/data file/ground_truth.csv'
IMG_DIR    = '/kaggle/input/image-classification-challenge-haradi/data file/img/test'
SAVE_DIR   = '/kaggle/working/'

IMG_SIZE   = 224
BATCH_SIZE = 32
NUM_EPOCHS = 35
LR         = 3e-4
N_FOLDS    = 5       # 5 models = 5 different data splits
SEED       = 42

MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

torch.manual_seed(SEED)
np.random.seed(SEED)
print(f'Will train {N_FOLDS} models — one per fold.')

Will train 5 models — one per fold.


In [3]:
# ── TRANSFORMS ─────────────────────────────────────────────
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.75, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.25),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
    transforms.RandomErasing(p=0.15),
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

print('Transforms ready.')

Transforms ready.


In [4]:
# ── DATASET ────────────────────────────────────────────────
class CatsDogsDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df        = df.reset_index(drop=True)
        self.img_dir   = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        path  = os.path.join(self.img_dir, f"{row['img']}.jpg")
        image = Image.open(path).convert('RGB')
        label = int(row['is_dog'])
        if self.transform:
            image = self.transform(image)
        return image, torch.tensor(label, dtype=torch.long)


df_all = pd.read_csv(CSV_PATH)
print(f'Total samples: {len(df_all)}')
print(df_all['is_dog'].value_counts())

Total samples: 1000
is_dog
0    514
1    486
Name: count, dtype: int64


In [5]:
# ── MODEL BUILDER ──────────────────────────────────────────
def build_model():
    model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
    num_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Linear(num_features, 256),
        nn.BatchNorm1d(256),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(256, 2)
    )
    return model.to(device)

print('Model builder ready.')

Model builder ready.


In [6]:
# ── K-FOLD TRAINING ────────────────────────────────────────
# Each fold: different 80% train / 20% val split
# Result: 5 diverse models that saw different data

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
fold_scores = []

X = df_all['img'].values
y = df_all['is_dog'].values

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f'\n{"="*55}')
    print(f'  FOLD {fold+1}/{N_FOLDS}  |  Train: {len(train_idx)}  |  Val: {len(val_idx)}')
    print(f'{"="*55}')

    df_train = df_all.iloc[train_idx]
    df_val   = df_all.iloc[val_idx]

    train_ds = CatsDogsDataset(df_train, IMG_DIR, train_transform)
    val_ds   = CatsDogsDataset(df_val,   IMG_DIR, val_transform)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

    # Fresh model for each fold
    model = build_model()

    backbone_params = [p for name, p in model.named_parameters() if 'fc' not in name]
    head_params     = list(model.fc.parameters())
    optimizer = optim.AdamW([
        {'params': backbone_params, 'lr': LR / 10},
        {'params': head_params,     'lr': LR},
    ], weight_decay=1e-4)

    scheduler = optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=[LR/10, LR],
        steps_per_epoch=len(train_loader),
        epochs=NUM_EPOCHS,
        pct_start=0.2,
    )

    criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
    scaler    = GradScaler()

    best_acc   = 0.0
    no_improve = 0
    save_path  = os.path.join(SAVE_DIR, f'fold_{fold+1}.pth')

    for epoch in range(NUM_EPOCHS):
        # TRAIN
        model.train()
        running_loss = 0.0
        for images, labels in tqdm(train_loader, desc=f'Fold {fold+1} Ep {epoch+1}/{NUM_EPOCHS}', leave=False):
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            with autocast():
                out  = model(images)
                loss = criterion(out, labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            running_loss += loss.item()

        # VALIDATE
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                with autocast():
                    out = model(images)
                _, pred = torch.max(out, 1)
                total   += labels.size(0)
                correct += (pred == labels).sum().item()

        val_acc = 100 * correct / total
        print(f'  Ep {epoch+1:02d} | Loss: {running_loss/len(train_loader):.4f} | Val: {val_acc:.2f}%')

        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), save_path)
            print(f'  ✅ Fold {fold+1} best saved ({best_acc:.2f}%)')
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= 10:
                print(f'  Early stop at epoch {epoch+1}')
                break

    fold_scores.append(best_acc)
    print(f'  🏆 Fold {fold+1} Best: {best_acc:.2f}%')
    del model
    torch.cuda.empty_cache()

print(f'\n{"="*55}')
print(f'All fold scores: {[f"{s:.2f}%" for s in fold_scores]}')
print(f'Average CV score: {np.mean(fold_scores):.2f}%')
print(f'{"="*55}')


  FOLD 1/5  |  Train: 800  |  Val: 200
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 219MB/s]
/tmp/ipykernel_24/2456295690.py:44: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler    = GradScaler()
Fold 1 Ep 1/35:   0%|          | 0/25 [00:00<?, ?it/s]/tmp/ipykernel_24/2456295690.py:57: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_24/2456295690.py:74: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


  Ep 01 | Loss: 0.7122 | Val: 81.50%
  ✅ Fold 1 best saved (81.50%)


  Ep 02 | Loss: 0.4191 | Val: 96.50%
  ✅ Fold 1 best saved (96.50%)


  Ep 03 | Loss: 0.2288 | Val: 97.50%
  ✅ Fold 1 best saved (97.50%)


...

  Ep 17 | Loss: 0.1275 | Val: 98.50%


  Ep 18 | Loss: 0.1300 | Val: 98.50%


  Ep 19 | Loss: 0.1385 | Val: 97.50%


  Ep 20 | Loss: 0.1271 | Val: 99.50%


  Ep 21 | Loss: 0.1291 | Val: 99.50%
  Early stop at epoch 21
  🏆 Fold 5 Best: 99.50%

All fold scores: ['99.00%', '99.50%', '99.00%', '99.50%', '99.50%']
Average CV score: 99.30%


In [7]:
# ── LOAD ALL 5 FOLD MODELS ─────────────────────────────────
fold_models = []
for fold in range(N_FOLDS):
    m = build_model()
    m.load_state_dict(torch.load(os.path.join(SAVE_DIR, f'fold_{fold+1}.pth')))
    m.eval()
    fold_models.append(m)
    print(f'Fold {fold+1} model loaded ✅')

print(f'\nAll {N_FOLDS} models loaded and ready for inference!')

Fold 1 model loaded ✅
Fold 2 model loaded ✅
Fold 3 model loaded ✅
Fold 4 model loaded ✅
Fold 5 model loaded ✅

All 5 models loaded and ready for inference!


In [8]:
# ── 12x TTA + 5-FOLD ENSEMBLE INFERENCE ───────────────────
# Total views per image = 12 TTA × 5 models = 60 perspectives!

def get_tta_transforms(size):
    norm = transforms.Normalize(MEAN, STD)
    return [
        transforms.Compose([transforms.Resize((size,size)), transforms.ToTensor(), norm]),
        transforms.Compose([transforms.Resize((size,size)), transforms.RandomHorizontalFlip(p=1), transforms.ToTensor(), norm]),
        transforms.Compose([transforms.Resize((int(size*1.15),int(size*1.15))), transforms.CenterCrop(size), transforms.ToTensor(), norm]),
        transforms.Compose([transforms.Resize((int(size*1.15),int(size*1.15))), transforms.CenterCrop(size), transforms.RandomHorizontalFlip(p=1), transforms.ToTensor(), norm]),
        transforms.Compose([transforms.Resize((size,size)), transforms.RandomRotation((10,10)), transforms.ToTensor(), norm]),
        transforms.Compose([transforms.Resize((size,size)), transforms.RandomRotation((-10,-10)), transforms.ToTensor(), norm]),
        transforms.Compose([transforms.Resize((size,size)), transforms.RandomRotation((10,10)), transforms.RandomHorizontalFlip(p=1), transforms.ToTensor(), norm]),
        transforms.Compose([transforms.Resize((size,size)), transforms.RandomRotation((-10,-10)), transforms.RandomHorizontalFlip(p=1), transforms.ToTensor(), norm]),
        transforms.Compose([transforms.Resize((int(size*1.3),int(size*1.3))), transforms.CenterCrop(size), transforms.ToTensor(), norm]),
        transforms.Compose([transforms.Resize((int(size*1.3),int(size*1.3))), transforms.CenterCrop(size), transforms.RandomHorizontalFlip(p=1), transforms.ToTensor(), norm]),
        transforms.Compose([transforms.Resize((int(size*1.1),int(size*1.1))), transforms.FiveCrop(size), transforms.Lambda(lambda c: c[0]), transforms.ToTensor(), norm]),
        transforms.Compose([transforms.Resize((int(size*1.1),int(size*1.1))), transforms.FiveCrop(size), transforms.Lambda(lambda c: c[4]), transforms.ToTensor(), norm]),
    ]

tta_transforms = get_tta_transforms(IMG_SIZE)
print(f'TTA transforms: {len(tta_transforms)}')
print(f'Total perspectives per image: {len(tta_transforms)} TTA × {N_FOLDS} models = {len(tta_transforms)*N_FOLDS} 🔥')

predictions = []

with torch.no_grad():
    for img_name in tqdm(sorted(os.listdir(IMG_DIR)), desc='5-Fold + TTA Inference'):
        if not img_name.endswith('.jpg'):
            continue
        image = Image.open(os.path.join(IMG_DIR, img_name)).convert('RGB')

        # Accumulate probs from ALL 5 models × 12 TTA = 60 predictions
        all_probs = torch.zeros(2).to(device)

        for fold_model in fold_models:
            fold_probs = torch.zeros(2).to(device)
            for tf in tta_transforms:
                t = tf(image).unsqueeze(0).to(device)
                with autocast():
                    out = fold_model(t)
                fold_probs += torch.softmax(out, dim=1).squeeze()
            fold_probs /= len(tta_transforms)
            all_probs += fold_probs

        all_probs /= N_FOLDS
        _, pred = torch.max(all_probs, 0)
        predictions.append((img_name.replace('.jpg',''), pred.item()))

print(f'\nTotal predictions: {len(predictions)}')

TTA transforms: 12
Total perspectives per image: 12 TTA × 5 models = 60 🔥


5-Fold + TTA Inference:   0%|          | 0/3000 [00:00<?, ?it/s]/tmp/ipykernel_24/4173420250.py:40: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
5-Fold + TTA Inference: 100%|██████████| 3000/3000 [28:31<00:00,  1.75it/s]


Total predictions: 3000


In [9]:
# ── SUBMISSION ─────────────────────────────────────────────
submission = pd.DataFrame(predictions, columns=['img', 'is_dog'])
submission['img'] = submission['img'].astype(int)
submission = submission.sort_values('img').reset_index(drop=True)
submission.to_csv('/kaggle/working/submission.csv', index=False)

print('✅ submission.csv saved!')
print(f"Dog: {submission['is_dog'].sum()} | Cat: {(submission['is_dog']==0).sum()}")
print(submission.head(10))

✅ submission.csv saved!
Dog: 1501 | Cat: 1499
   img  is_dog
0    0       1
1    1       0
2    2       1
3    3       0
4    4       1
5    5       0
6    6       0
7    7       1
8    8       0
9    9       1
